Using data from wheater.com. The robot.txt says that all user-agents are allowed, as long as the rate-limit is at least 10. No explicit visit-time. The Requests - agent does not seem to be disallowed

# Write the Class

In [ ]:
import requests # scrapping
from bs4 import BeautifulSoup # parsing
import datetime # crawl-delay, get todays date
import time
import locale # translate date from english to german
import sys
locale.setlocale(locale.LC_ALL, 'deu_deu')

import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd

class CustomException(Exception):
    def __init__(self,msg):
        self.msg=msg
        print('Requests error occured')

class Temperature_Scrapper:
    def __init__(self):
        self.url_current = "https://weather.com/de-DE/wetter/stundlich/l/162d176a76330db38e03f852479051d26cc80e81f7a0da9a4efb1ea45f8a4ed2" # Contains the hourly temperature for today, tommorrow and the day after tommorrow
        self.url_std = "https://weather.com/de-DE/wetter/heute/l/162d176a76330db38e03f852479051d26cc80e81f7a0da9a4efb1ea45f8a4ed2" # contains (among other things) the expected temperature range for the same days as the first link
        self.Temp_dict = {"Current": {},
                         "Std": {}} # Store the hourly temperature and the range

    def _scrap(self):
        """
        Scrap Weather.com for the hourly temperature data and close in on the part of the hml containing that information
        """
        response = requests.get(self.url_current)
        if response.status_code != 200:
            return response.status_code
        soup = BeautifulSoup(response.text, "html.parser")
        entry_containers = soup.find_all("div", class_ = "DetailsSummary--DetailsSummary--Mt7BE DetailsSummary DetailsSummary--hourlyDetailsSummary--i0dUl")
        return entry_containers

    def _fish_current(self):
        """
        Search in the scrapped url for the temperature data of today and the next 48 hours and save it.
        """
        Date = datetime.datetime.today()
        response = self._scrap()
        if response is int and response != 200:
            raise CustomException(f"Error: Request status code {response}")
            sys.exit()

        for entry in response:
            time_of_day = entry.select("h2")[0].text # select the first h2 element to find the time
            if len(time_of_day) < 5:
                time_of_day = "0" + time_of_day
            
            if time_of_day == "00:00":
                Date += datetime.timedelta(days = 1)
            date_of_day = Date.date()
            
            css_selector = "div:nth-child(3) > span:nth-child(1)"  # select the thrid element (which should be a div) and of that, select the first element (which should be a span) to get the temperature in °C.
            temperature = int(entry.select(css_selector)[0].text.replace("°", ""))
        

            date_and_time= datetime.datetime(int(date_of_day.strftime("%Y")), int(date_of_day.strftime("%m")), int(date_of_day.strftime("%d")), int(time_of_day.split(":")[0]), 0)
            self.Temp_dict["Current"][date_and_time] = temperature

    def _fish_std(self):
        """
        Scrape for the temperature range and sacv it
        """
        response = requests.get(self.url_std)
        if response.status_code != 200:
            return response.status_code
        soup = BeautifulSoup(response.text, "html.parser")
        daily_overview = soup.find("div", id = "WxuDailyWeatherCard-main-bb1a17e7-dc20-421a-b1b8-c117308c6626")


        day = {"Heute": "section:nth-child(1) > div:nth-child(2) > div:nth-child(1) > ul:nth-child(1) > li:nth-child(1) > a:nth-child(1) > div:nth-child(2)",
              "Morgen": "section:nth-child(1) > div:nth-child(2) > div:nth-child(1) > ul:nth-child(1) > li:nth-child(2) > a:nth-child(1) > div:nth-child(2)",
              "Übermorgen": "section:nth-child(1) > div:nth-child(2) > div:nth-child(1) > ul:nth-child(1) > li:nth-child(3) > a:nth-child(1) > div:nth-child(2)"}
        

        span = {"Max": " > span:nth-child(1)",
               "Min": " > span:nth-child(2) > span:nth-child(1)"}
        
        for i in day:
            self.Temp_dict["Std"].setdefault(i, {})
            for j in span:
                css_selector = day[i] + span[j]
                value = int(daily_overview.select(css_selector)[0].text.replace("°", ""))
                self.Temp_dict["Std"][i][j] = value
                
                

    def get_dict(self):
        return self.Temp_dict

    def prep_plotting(self):
        """ Transform the Data for plotting"""
        heute = datetime.datetime.today().strftime("%A")
        morgen = (datetime.datetime.today() + datetime.timedelta(1)).strftime("%A")
        übermorgen = (datetime.datetime.today() + datetime.timedelta(2)).strftime("%A")
        
        day_dict = {heute: "Heute",
                   morgen: "Morgen",
                   übermorgen: "Übermorgen"}
        
        
        self.main_df =  pd.DataFrame.from_dict(self.Temp_dict["Current"], orient = "index").reset_index()
        self.main_df.columns = ["DateTime", "Temp [°C]"]
        self.main_df["DayOfWeek"] = self.main_df.loc[:, "DateTime"].apply(lambda x: day_dict[x.strftime("%A")])
        self.main_df["TimeOfDay"] = self.main_df.loc[:, "DateTime"].apply(lambda x: x.strftime("%H:%M"))
        self.main_df.sort_values(by = "DateTime", inplace = True)

        self.sup_df = pd.DataFrame.from_dict(self.Temp_dict["Std"]).transpose()

        
    def plot_temperature(self, period, update_counter = 0):
        """
        Create a line+scatterplot of the hourly data with the dataranges as coloured boxes in the background.
        You can choose whether to plot just today, today and tommorrow or all the data that was scrapped.
        """
        mask_dict = {"Heute": self.main_df.loc[:, "DayOfWeek"] == "Heute",
                    "Morgen": np.logical_or(self.main_df.loc[:, "DayOfWeek"] == "Heute", self.main_df.loc[:, "DayOfWeek"] == "Morgen"),
                    "Übermorgen": np.logical_or.reduce((self.main_df.loc[:, "DayOfWeek"] == "Heute", self.main_df.loc[:, "DayOfWeek"] == "Morgen", self.main_df.loc[:, "DayOfWeek"] == "Übermorgen"))}
    
        mask = mask_dict[period]
        main_masked = self.main_df.loc[mask, :]
    
        ###############################################
        plot_data = go.Scatter(x = list(range(main_masked.shape[0])), y = main_masked.loc[:, "Temp [°C]"], mode = "lines+markers", name = "Temperature [°C]", legendgroup="group1", legendgrouptitle_text="Hourly data", showlegend=True)
        plot = go.Figure(data = [plot_data])
        
        # Rectangles showing the variation in temperature for a day
        # Vertical Lines signafying the end of a day

        colors = ["black", "purple", "yellow"]
        
        for i, day in enumerate(np.unique(main_masked["DayOfWeek"])):
            mask = main_masked["DayOfWeek"] == day
            x0, x1 = main_masked.loc[mask, :].index[0], main_masked.loc[mask, :].index[-1]
            y0, y1 = self.sup_df.loc[day, "Min"], self.sup_df.loc[day, "Max"]

            if i == 0:
                plot.add_shape(x0 = x0, x1 = x1, y0 = y0, y1 = y1, layer = "below", fillcolor = colors[i], opacity = 0.2, name = list(mask_dict.keys())[i], legendgroup="group2", legendgrouptitle_text="Expected temperature range", showlegend=True)
            else:
                plot.add_shape(x0 = x0, x1 = x1, y0 = y0, y1 = y1, layer = "below", fillcolor = colors[i], opacity = 0.2, name = list(mask_dict.keys())[i], legendgroup="group2", showlegend=True)

        plot.update_layout(
            title = dict(text = f"Times graph got updated since application start: {update_counter}"),
            xaxis = dict( tickmode = 'array', tickvals = list(range(0, main_masked.shape[0], 2)), ticktext = [main_masked.loc[i, "TimeOfDay"] for i in range(0, main_masked.shape[0], 2)], title=dict(text= "Time of Day")),
            yaxis = dict(title = dict(text = "Temperature [°C]")),
            # legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
        )
        return plot

    def startup(self):
        """
        Perform all the scrapping and data prepataion
        """
        self._fish_current()
        self._fish_std()
        self.prep_plotting()

    def test1(self):
        """
        Test Scrapping of Hourly Data
        """
        self._fish_current()

    def test2(self):
        """
        Test Scrapping of Overall Range
        """
        self._fish_std()

    def test3(self):
        """
        Test data prepatation for plotting and the plotting itself.
        """
        self.prep_plotting()
        plot = self.plot_temperature("Übermorgen")
        return plot

In [ ]:
testing = False
if testing:
    webscrapper = Temperature_Scrapper()
    webscrapper.test1()
    time.sleep(10) # according to Weather.coms robot.txt, there should be at least 10 seconds between requests
    webscrapper.test2()
    webscrapper.test3()

In [ ]:
from dash import Dash, html, dash_table, dcc, callback, Output, Input

### StartUp
webscrapper = Temperature_Scrapper()
webscrapper.startup()
period = "Heute"
update_counter = 0


#### The Interactive Plot
app = Dash()
app.layout = html.Div([
        html.H4('Weather.com - Temperature'),
        dcc.Graph(id='live-update-graph', figure = webscrapper.plot_temperature("Heute")),
        dcc.Dropdown(['Heute', 'Morgen', 'Übermorgen'], 'Heute', id='Day_Dropdown'), # A callback is periodically fired
        dcc.Store(id = "PeriodStorage"),
        dcc.Interval(id = "UpdateCounter", interval = 15000)
    ])


##### The callbacks
@callback( Output('live-update-graph', 'figure', allow_duplicate=True), Input('Day_Dropdown', 'value'), prevent_initial_call = True)
def change_graph(value):
    """Replace the current graph"""
    return webscrapper.plot_temperature(value, update_counter)

@callback( Output('PeriodStorage', 'data'), Input('Day_Dropdown', 'value'), )
def save_period(value):
    """Save the current period"""
    global period
    period = value
    return {"period": value}
    
@callback( Output("live-update-graph", "figure", allow_duplicate=True), Input("UpdateCounter", "n_intervals"), prevent_initial_call = True )
def update_Scrapper(n_intervals):
    """
    Scrape the website again and update the values
    """
    global update_counter
    update_counter += 1
    global webscrapper
    webscrapper.startup()
    plot = webscrapper.plot_temperature(period, update_counter = update_counter)
    return plot



###### Start the Application
if __name__ == '__main__':
    app.run(debug=True)